## Setup and Configuration

In [ ]:
import numpy as np
import os
import pandas as pd
from sklearn.decomposition import PCA
from itertools import product
from collections import defaultdict
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add project root to path
ROOT = Path.cwd().parents[1]  # Assumes notebook is in projects/MOSAIC/notebooks/
sys.path.append(str(ROOT))
sys.path.append(str(ROOT / "src"))

# Import MOSAIC-specific utilities
from pose_dynamics.projects.MOSAIC import (
    extract_keypoints,
    load_mosaic_session,
    load_trial_info,
    extract_audio_conditions,
    preprocess_mosaic_trial,
    get_window_indices,
    build_symmetric_template,
    compute_reference_limb_lengths,
    batch_apply_fixed_lengths,
    align_keypoints,
    rebuild_aligned_dataframe,
    add_custom_features,
    compute_velocity,
    compute_linear_metrics
)

from pose_dynamics.projects.MOSAIC.alignment import order_xy_pairs
from pose_dynamics.projects.MOSAIC.visualization import (
    plot_alignment_diagnostics,
    create_pc_animation
)

# Import shared utilities
from pose_dynamics.preprocessing import (
    normalize_by_resolution,
    mask_low_confidence,
    interpolate_nans,
    filter_data_safe_preserve_nans
)

# Import RQA utilities (you'll need to ensure these are available)
# from utils.rqa_utils import perform_crqa, perform_rqa

print("✓ Imports successful")

## Parameters

In [ ]:
# === Data Configuration ===
DATA_PATH = "G:/mosaic analysis files"  # Update to your data path
KEYPOINT_SETS = [["center_face", "body", "arm"]]

# === Processing Parameters ===
CONF_THRESHOLD = 0.4
MAX_INTERP_GAP = 60
FILTER_CUTOFF = 10.0
FILTER_ORDER = 4
FPS = 60

# === Window Parameters ===
WINDOW_SIZE_RQA = 60 * 60      # 60 seconds for RQA/CRQA
WINDOW_SIZE_LINEAR = 5 * 60    # 5 seconds for linear metrics
WINDOW_OVERLAP = 0.5

# === PCA Parameters ===
N_COMPONENTS = 6

# === RQA/CRQA Parameters ===
TIME_LAGS = [15]
EMBEDDING_DIMS = [4]
RADII = [0.15]

# === Alignment Parameters ===
SYMMETRIZATION_MODE = "none"  # Options: "none", "nose", "torso", "full"
ALLOW_ROTATION = True
USE_VELOCITY = False  # Set True to analyze velocity instead of position

# === Session Selection ===
SKIP_SESSIONS = [1]  # Sessions to skip

# === Output Directories ===
os.makedirs("../crqa_results", exist_ok=True)
os.makedirs("../rqa_results", exist_ok=True)
os.makedirs("../linear_results", exist_ok=True)
os.makedirs("../animations", exist_ok=True)

print("✓ Parameters configured")

## Helper Functions

In [ ]:
def safe_get(dct, key):
    """Safely extract value from dict, return NaN if missing."""
    return dct.get(key, np.nan) if dct else np.nan


def display_summary(trials, expected_cols):
    """Display summary statistics for loaded data."""
    n_trials = len(trials)
    sessions = {meta['Session'] for _, meta in trials}
    conditions = {meta['Condition'] for _, meta in trials}
    roles = {meta['Role'] for _, meta in trials}
    
    print(f"\n{'='*60}")
    print("DATA SUMMARY")
    print(f"{'='*60}")
    print(f"Total trials loaded:      {n_trials}")
    print(f"Sessions:                 {len(sessions)} (S{min(sessions):03} - S{max(sessions):03})")
    print(f"Unique conditions:        {len(conditions)} {sorted(conditions)}")
    print(f"Roles:                    {sorted(roles)}")
    print(f"Keypoints per trial:      {len(expected_cols) // 2}")
    print(f"{'='*60}\n")

## Pass 1: Load Data and Build Global Template

This pass:
1. Loads all session/trial pose files
2. Extracts relevant keypoints
3. Preprocesses (normalize, mask, interpolate, filter)
4. Slices into windows
5. Builds global symmetric template from all data

In [ ]:
raw_windows = []   # (window_df, metadata)
full_trials = []   # for template building
expected_cols = None

print("Loading and preprocessing data...")

for session_number in range(2, 3):
    if session_number in SKIP_SESSIONS:
        print(f"[SKIP] Session {session_number:03} (user request)")
        continue

    # Load trial info
    trial_info = load_trial_info(DATA_PATH, session_number)
    if trial_info is None:
        continue
    
    audio_conditions = extract_audio_conditions(trial_info)

    for trial_number in range(1, 7):
        for role, suffix in [("Left", "left"), ("Right", "right")]:
            # Load pose data
            data = load_mosaic_session(DATA_PATH, session_number, trial_number, suffix)
            if data is None:
                continue

            # Extract keypoints
            selected = extract_keypoints(data, sets=KEYPOINT_SETS[0])
            
            # Get expected columns from first valid trial
            if expected_cols is None and not selected.empty:
                expected_cols = order_xy_pairs(selected.columns)
                print(f"✓ Column order established: {len(expected_cols)} columns ({len(expected_cols)//2} keypoints)")
            
            # Preprocess
            try:
                preprocessed = preprocess_mosaic_trial(
                    selected,
                    expected_cols,
                    conf_threshold=CONF_THRESHOLD,
                    max_interp_gap=MAX_INTERP_GAP,
                    filter_cutoff=FILTER_CUTOFF,
                    filter_order=FILTER_ORDER,
                    fps=FPS
                )
                
                # Keep full trial for template
                full_trials.append(preprocessed)
                
                # Create windows
                for ws in [WINDOW_SIZE_RQA]:
                    win_indices = get_window_indices(len(preprocessed), ws, WINDOW_OVERLAP)
                    for w_idx, (start, end) in enumerate(win_indices):
                        window = preprocessed.iloc[start:end].reset_index(drop=True)
                        
                        meta = {
                            'Session': session_number,
                            'Trial': trial_number,
                            'Role': role,
                            'Condition': audio_conditions[trial_number - 1],
                            'Window_Size': ws,
                            'Window_Index': w_idx
                        }
                        
                        # Check for NaNs
                        if window.isnull().any().any() or len(window) < ws:
                            raw_windows.append((None, meta))
                        else:
                            raw_windows.append((window, meta))
                            
            except Exception as e:
                print(f"Error processing S{session_number:03} T{trial_number} {role}: {e}")
                continue

print(f"\n✓ Loaded {len([w for w, _ in raw_windows if w is not None])} valid windows from {len(full_trials)} trials")

# Build global template
if not full_trials:
    raise ValueError("No valid trials found for template building")

X_raw = pd.concat(full_trials, ignore_index=True).dropna().astype(np.float32)
global_template = build_symmetric_template(X_raw, expected_cols, mode=SYMMETRIZATION_MODE)
print(f"✓ Global template built: {global_template.shape} with mode='{SYMMETRIZATION_MODE}'")

# Compute reference limb lengths
ref_lengths = None if SYMMETRIZATION_MODE == "nose" else compute_reference_limb_lengths(global_template, expected_cols)
if ref_lengths:
    print(f"✓ Reference limb lengths computed: {len(ref_lengths)} segments")

## Alignment Diagnostics

Visualize alignment quality by comparing raw, aligned, and limb-rescaled poses.

In [ ]:
# Sample windows from both Left and Right roles for better visualization
valid_windows = [w for w in raw_windows if w[0] is not None]

# Separate by role
left_windows = [(w, m) for w, m in valid_windows if m['Role'] == 'Left']
right_windows = [(w, m) for w, m in valid_windows if m['Role'] == 'Right']

# Take samples from each role
sample_windows = []
n_per_role = 5
if left_windows:
    sample_windows.extend(left_windows[:n_per_role])
if right_windows:
    sample_windows.extend(right_windows[:n_per_role])

plot_alignment_diagnostics(
    global_template=global_template,
    raw_windows=sample_windows,
    expected_cols=expected_cols,
    align_keypoints=align_keypoints,
    n_samples=len(sample_windows),
    procrustes=True,
    allow_rotation=ALLOW_ROTATION,
    ref_lengths=ref_lengths,
    reference="Torso"
)

## Pass 2: Align Windows and Prepare for Analysis

This pass:
1. Aligns all windows to global template using Procrustes
2. Applies limb length constraints
3. Optionally computes velocity
4. Centers data per trial (removes camera angle effects)

In [ ]:
aligned_windows = []
metadata = []

print("Aligning windows...")

for window, meta in raw_windows:
    if window is None:
        aligned_windows.append(None)
        metadata.append(meta)
        continue

    # Align to global template (WITH scale normalization to match old code)
    aligned_X, _ = align_keypoints(
        window, expected_cols,
        reference="Torso" if SYMMETRIZATION_MODE != "nose" else "Nose",
        template=global_template,
        use_procrustes=True,
        allow_rotation=ALLOW_ROTATION,
        allow_scale=True  # Match old code: Procrustes WITH scale
    )

    # Apply limb length constraints (matching old code)
    poses = aligned_X.reshape(-1, len(expected_cols)//2, 2)
    if ref_lengths:
        poses = batch_apply_fixed_lengths(poses, ref_lengths)

    # Rebuild DataFrame
    aligned_df = rebuild_aligned_dataframe(
        poses.reshape(-1, len(expected_cols)), expected_cols
    )

    # Optionally compute velocity
    if USE_VELOCITY:
        aligned_df = compute_velocity(aligned_df, fps=FPS)

    aligned_windows.append(aligned_df)
    metadata.append(meta)

print(f"✓ Aligned {len([w for w in aligned_windows if w is not None])} windows")

# Trial-centered normalization (removes camera angle effects)
trial_groups = defaultdict(list)
for i, meta in enumerate(metadata):
    if aligned_windows[i] is None:
        continue
    key = (meta['Session'], meta['Trial'], meta['Role'])
    trial_groups[key].append((i, aligned_windows[i]))

trial_centered_windows = [None] * len(aligned_windows)
for indices_and_data in trial_groups.values():
    trial_df = pd.concat([df for _, df in indices_and_data], ignore_index=True)
    trial_mean = trial_df.mean()
    for idx, df in indices_and_data:
        trial_centered_windows[idx] = df - trial_mean

print("✓ Trial-centered normalization complete")

## PCA Analysis

Fit PCA on trial-centered data to identify principal movement patterns.

In [ ]:
# Stack trial-centered data for PCA
stacked_trial = pd.concat(
    [df for df in trial_centered_windows if df is not None], 
    ignore_index=True
).astype(np.float32)

# Diagnostic information
print(f"\n{'='*60}")
print("PCA INPUT DATA DIAGNOSTICS")
print(f"{'='*60}")
print(f"Stacked data shape: {stacked_trial.shape}")
print(f"Number of windows: {len([w for w in trial_centered_windows if w is not None])}")
print(f"Frames per window: {WINDOW_SIZE_RQA}")
print(f"Data range - min: {stacked_trial.min().min():.4f}, max: {stacked_trial.max().max():.4f}")
print(f"Data std dev: {stacked_trial.std().mean():.4f}")
print(f"NaN count: {stacked_trial.isnull().sum().sum()}")
print(f"Inf count: {np.isinf(stacked_trial.values).sum()}")
print(f"{'='*60}\n")

if N_COMPONENTS > stacked_trial.shape[1]:
    N_COMPONENTS = stacked_trial.shape[1]
    print(f"⚠ Reduced n_components to {N_COMPONENTS} (max features)")

# Fit PCA
pca = PCA(n_components=N_COMPONENTS).fit(stacked_trial)

# Display variance explained
print(f"\n{'='*60}")
print("PCA VARIANCE EXPLAINED")
print(f"{'='*60}")
for i, var in enumerate(pca.explained_variance_ratio_):
    cum_var = pca.explained_variance_ratio_[:i+1].sum()
    print(f"PC{i+1}: {var*100:5.2f}%  (cumulative: {cum_var*100:5.2f}%)")
print(f"{'='*60}\n")

# Project windows into PCA space
for i, meta in enumerate(metadata):
    if trial_centered_windows[i] is None:
        meta['pcs'] = None
    else:
        meta['pcs'] = pca.transform(trial_centered_windows[i])

print(f"✓ Projected {sum(1 for m in metadata if m['pcs'] is not None)} windows into PC space")

## PCA Animation

Generate animations showing movement patterns for each principal component.

In [ ]:
# Stack global-centered data for animation (better visualization)
stacked_global = pd.concat(
    [df for df in aligned_windows if df is not None], 
    ignore_index=True
).astype(np.float32)
global_mean_vector = stacked_global.mean().values
global_centered = stacked_global - global_mean_vector

# Create animation
create_pc_animation(
    pca=pca,
    X=global_centered,
    keypoint_names=expected_cols,
    save_path="../animations/mosaic_pc_anims.mp4",
    mean_vector=global_mean_vector,
    scale=2,
    n_components=min(6, N_COMPONENTS),
    mode="sine",
    fps=40,
    ref_lengths=ref_lengths,
    zoom_factor=1.0
)

print("✓ Animation saved to ../animations/mosaic_pc_anims.mp4")

## CRQA Analysis: Dyadic Synchrony (PCs)

Analyze interpersonal coordination between Left and Right participants using cross-recurrence on principal components.

In [ ]:
# Pair Left/Right windows
paired_windows = defaultdict(dict)
for m in metadata:
    key = (m['Session'], m['Trial'], m['Window_Index'])
    paired_windows[key][m['Role']] = m

csv_rows = []
n_pcs = min(N_COMPONENTS, pca.n_components_)

print(f"Running CRQA on {len(paired_windows)} window pairs...")

for key, pair in paired_windows.items():
    if "Left" in pair and "Right" in pair:
        left_meta, right_meta = pair["Left"], pair["Right"]
        left_pcs, right_pcs = left_meta['pcs'], right_meta['pcs']
        
        if left_pcs is None or right_pcs is None:
            continue

        for lag, emb_dim, rad in product(TIME_LAGS, EMBEDDING_DIMS, RADII):
            for pc_idx in range(min(N_COMPONENTS, left_pcs.shape[1], right_pcs.shape[1])):
                # Prepare CRQA input
                crqa_input = pd.DataFrame({
                    'left': left_pcs[:, pc_idx],
                    'right': right_pcs[:, pc_idx]
                })
                
                # Run CRQA (you'll need to implement or import this)
                # crqa_out, _ = perform_crqa(crqa_input, {
                #     'eDim': emb_dim, 'tLag': lag, 'radius': rad,
                #     'minl': 2, 'tw': 0, 'norm': 2,
                #     'rescaleNorm': 1, 'showMetrics': False,
                #     'doStatsFile': False, 'plotMode': 'none', 'saveFig': False
                # }, f"S{key[0]:03}_T{key[1]}_W{key[2]}_PC{pc_idx+1}.csv")
                
                # For now, create placeholder
                crqa_out = {
                    'perc_recur': np.nan,
                    'perc_determ': np.nan,
                    'mean_line_length': np.nan,
                    'maxl_found': np.nan,
                    'entropy': np.nan,
                    'laminarity': np.nan,
                    'trapping_time': np.nan
                }
                
                rec_val = safe_get(crqa_out, 'perc_recur')
                if pc_idx == 0:  # Print progress for PC1
                    print(f"[CRQA] S{key[0]:03}, T{key[1]}, W{key[2]}, PC1 -> REC={rec_val:.2f}%")

                csv_rows.append({
                    'Session': key[0],
                    'Trial': key[1],
                    'Condition': left_meta['Condition'],
                    'Window_Size': left_meta['Window_Size'],
                    'Window_Index': key[2],
                    'Lag': lag,
                    'Embedding_Dim': emb_dim,
                    'Radius': rad,
                    'PC': pc_idx + 1,
                    'CRQA_REC': safe_get(crqa_out, 'perc_recur'),
                    'CRQA_DET': safe_get(crqa_out, 'perc_determ'),
                    'CRQA_Lmean': safe_get(crqa_out, 'mean_line_length'),
                    'CRQA_Lmax': safe_get(crqa_out, 'maxl_found'),
                    'CRQA_ENT': safe_get(crqa_out, 'entropy'),
                    'CRQA_LAM': safe_get(crqa_out, 'laminarity'),
                    'CRQA_TT': safe_get(crqa_out, 'trapping_time')
                })
    else:
        # Missing role → NaNs
        meta_any = pair.get("Left", pair.get("Right"))
        for lag, emb_dim, rad in product(TIME_LAGS, EMBEDDING_DIMS, RADII):
            for pc_idx in range(n_pcs):
                csv_rows.append({
                    'Session': key[0],
                    'Trial': key[1],
                    'Condition': meta_any['Condition'] if meta_any else None,
                    'Window_Size': meta_any['Window_Size'] if meta_any else None,
                    'Window_Index': key[2],
                    'Lag': lag,
                    'Embedding_Dim': emb_dim,
                    'Radius': rad,
                    'PC': pc_idx + 1,
                    'CRQA_REC': np.nan,
                    'CRQA_DET': np.nan,
                    'CRQA_Lmean': np.nan,
                    'CRQA_Lmax': np.nan,
                    'CRQA_ENT': np.nan,
                    'CRQA_LAM': np.nan,
                    'CRQA_TT': np.nan
                })

# Save CRQA results
out_path = f"../crqa_results/crqa_pc_R{int(RADII[0]*100)}.csv"
pd.DataFrame(csv_rows).to_csv(out_path, index=False)
print(f"✓ CRQA (PC) results saved → {out_path}")

## RQA Analysis: Individual Temporal Structure (PCs)

Analyze recurrence patterns within each individual's principal component trajectories.

In [ ]:
csv_rows = []

print(f"Running RQA on {len(metadata)} windows...")

for meta in metadata:
    pcs = meta['pcs']
    
    for lag, emb_dim, rad in product(TIME_LAGS, EMBEDDING_DIMS, RADII):
        for pc_idx in range(N_COMPONENTS):
            if pcs is None:
                # Missing window → NaN row
                csv_rows.append({
                    'Session': meta['Session'],
                    'Trial': meta['Trial'],
                    'Role': meta['Role'],
                    'Condition': meta['Condition'],
                    'Window_Size': meta['Window_Size'],
                    'Window_Index': meta['Window_Index'],
                    'Lag': lag,
                    'Embedding_Dim': emb_dim,
                    'Radius': rad,
                    'PC': pc_idx + 1,
                    'RQA_REC': np.nan,
                    'RQA_DET': np.nan,
                    'RQA_Lmean': np.nan,
                    'RQA_Lmax': np.nan,
                    'RQA_pLmax': np.nan,
                    'RQA_ENT': np.nan,
                    'RQA_LAM': np.nan,
                    'RQA_TT': np.nan,
                })
            else:
                # Run RQA
                rqa_input = pd.DataFrame({'x': pcs[:, pc_idx]})
                
                # rqa_out, _ = perform_rqa(rqa_input, {
                #     'eDim': emb_dim, 'tLag': lag, 'radius': rad,
                #     'minl': 2, 'tw': 2, 'norm': 2,
                #     'rescaleNorm': 1, 'showMetrics': False,
                #     'doStatsFile': False, 'plotMode': 'none',
                #     'saveFig': False, 'pointSize': 1
                # }, f"S{meta['Session']:03}_T{meta['Trial']}_W{meta['Window_Index']}_PC{pc_idx+1}.csv")
                
                # Placeholder
                rqa_out = {
                    'perc_recur': np.nan,
                    'perc_determ': np.nan,
                    'mean_line_length': np.nan,
                    'maxl_found': np.nan,
                    'entropy': np.nan,
                    'laminarity': np.nan,
                    'trapping_time': np.nan,
                }
                
                rec_val = safe_get(rqa_out, 'perc_recur')
                if pc_idx == 0 and meta['Window_Index'] == 0:  # Print first window, PC1
                    print(f"[RQA] S{meta['Session']:03} T{meta['Trial']} {meta['Role']} PC1 -> REC={rec_val:.2f}%")

                lmax = safe_get(rqa_out, 'maxl_found')
                pLmax = (lmax / meta['Window_Size']) if (isinstance(lmax, (int, float)) and meta['Window_Size']) else np.nan

                csv_rows.append({
                    'Session': meta['Session'],
                    'Trial': meta['Trial'],
                    'Role': meta['Role'],
                    'Condition': meta['Condition'],
                    'Window_Size': meta['Window_Size'],
                    'Window_Index': meta['Window_Index'],
                    'Lag': lag,
                    'Embedding_Dim': emb_dim,
                    'Radius': rad,
                    'PC': pc_idx + 1,
                    'RQA_REC': safe_get(rqa_out, 'perc_recur'),
                    'RQA_DET': safe_get(rqa_out, 'perc_determ'),
                    'RQA_Lmean': safe_get(rqa_out, 'mean_line_length'),
                    'RQA_Lmax': safe_get(rqa_out, 'maxl_found'),
                    'RQA_pLmax': pLmax,
                    'RQA_ENT': safe_get(rqa_out, 'entropy'),
                    'RQA_LAM': safe_get(rqa_out, 'laminarity'),
                    'RQA_TT': safe_get(rqa_out, 'trapping_time'),
                })

# Save RQA results
out_path = f"../rqa_results/rqa_pc_R{int(RADII[0]*100)}.csv"
pd.DataFrame(csv_rows).to_csv(out_path, index=False)
print(f"✓ RQA (PC) results saved → {out_path}")

## Linear Metrics Analysis (PCs)

Compute velocity, acceleration, and RMS metrics for each principal component.

In [ ]:
# Reload windows with shorter window size for linear metrics
linear_windows = []
linear_metadata = []

print("Creating windows for linear metrics...")

for session_number in range(1, 50):
    if session_number in SKIP_SESSIONS:
        continue

    trial_info = load_trial_info(DATA_PATH, session_number)
    if trial_info is None:
        continue
    
    audio_conditions = extract_audio_conditions(trial_info)

    for trial_number in range(1, 7):
        for role in ["Left", "Right"]:
            # Find the preprocessed trial data from full_trials
            # This is simplified - in practice you'd re-load or cache
            continue  # Skip for now - would need to store trial-level data

# For now, compute linear metrics on existing PC projections
csv_rows = []

for meta in metadata:
    pcs = meta['pcs']
    
    for pc_idx in range(N_COMPONENTS):
        if pcs is None:
            csv_rows.append({
                'Session': meta['Session'],
                'Trial': meta['Trial'],
                'Role': meta['Role'],
                'Condition': meta['Condition'],
                'Window_Size': meta['Window_Size'],
                'Window_Index': meta['Window_Index'],
                'PC': pc_idx + 1,
                'RMS': np.nan,
                'MeanVel': np.nan,
                'StdVel': np.nan,
                'MeanAcc': np.nan,
                'StdAcc': np.nan,
                'MeanVelMag': np.nan,
                'StdVelMag': np.nan,
                'MeanAccelMag': np.nan,
                'StdAccelMag': np.nan
            })
        else:
            series = pcs[:, pc_idx]
            metrics = compute_linear_metrics(series, fps=FPS)
            csv_rows.append({
                'Session': meta['Session'],
                'Trial': meta['Trial'],
                'Role': meta['Role'],
                'Condition': meta['Condition'],
                'Window_Size': meta['Window_Size'],
                'Window_Index': meta['Window_Index'],
                'PC': pc_idx + 1,
                **metrics
            })

out_path = "../linear_results/linear_pc.csv"
pd.DataFrame(csv_rows).to_csv(out_path, index=False)
print(f"✓ Linear metrics (PC) saved → {out_path}")